# 04 — Disambiguation

Resolve each extracted toponym to a stable geographic identifier — a Pleiades
URI (preferred, ancient-places focused) or, as a fallback, a GeoNames ID —
through a four-pass pipeline:

1. **Exact match** on the Pleiades name index
2. **Fuzzy match** (RapidFuzz) on the Pleiades name index
3. **LLM arbitration** among multiple Pleiades candidates
4. **GeoNames fallback** with LLM arbitration, for toponyms Pleiades cannot resolve

The output is one record **per toponym instance** (not per article), ready for
`05_output.ipynb` and `06_evaluation.ipynb`. The notebook is checkpoint/resume
aware: articles already present in the output file are skipped on re-run.

## CONFIG

In [1]:
import json
import logging
import pickle
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import jsonlines
import ollama
import requests
from rapidfuzz import process
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("aec_geoparser.disambiguation")

NER_FILE         = Path("../data/results/ner_results.jsonl")
ARTICLES_FILE    = Path("../data/cache/articles_with_abstract.jsonl")
INDEX_FILE       = Path("../data/cache/pleiades_index.pkl")
OUTPUT_FILE      = Path("../data/results/disambiguated.jsonl")
FUZZY_THRESHOLD  = 85       # rapidfuzz score cutoff (0-100)
FUZZY_CANDIDATES = 3        # max candidates sent to LLM arbitration
OLLAMA_MODEL     = "qwen2.5:32b"
OLLAMA_HOST      = "http://localhost:11434"
GEONAMES_USER    = "giacomo.mancuso"  # replace with your GeoNames username
GEONAMES_URL     = "http://api.geonames.org/searchJSON"
GEONAMES_FCLS    = ["P", "H", "T", "L", "S"]
GEONAMES_ROWS    = 5

USER_AGENT = "AeC-Geoparser/1.0 (github.com/gmancuso24/AeC_geoparser)"
TIMEOUT_S  = 30

client = ollama.Client(host=OLLAMA_HOST)


/Users/ijack/Documents/GitHub/AeC_geoparser/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Helpers

`normalize` builds diacritic- and case-insensitive lookup keys, identical to the one used to build the Pleiades index in `03_gazetteer.ipynb`. `search_geonames` is redefined here with identical logic so this notebook can run independently.

In [2]:
def normalize(s: str) -> str:
    """Lowercase and strip diacritics to obtain a stable lookup key."""
    return unicodedata.normalize("NFKD", s.lower()).encode("ascii", "ignore").decode()


_geonames_cache: dict[str, list[dict]] = {}


def search_geonames(toponym: str) -> list[dict]:
    """Query the GeoNames API for a toponym, returning up to GEONAMES_ROWS candidates."""
    if toponym in _geonames_cache:
        return _geonames_cache[toponym]

    params = {
        "q": toponym,
        "maxRows": GEONAMES_ROWS,
        "featureClass": GEONAMES_FCLS,
        "username": GEONAMES_USER,
    }
    headers = {"User-Agent": USER_AGENT}
    try:
        response = requests.get(GEONAMES_URL, params=params, headers=headers, timeout=TIMEOUT_S)
        response.raise_for_status()
        payload = response.json()
    except requests.exceptions.RequestException as exc:
        logger.error("GeoNames query failed for %r: %s", toponym, exc)
        _geonames_cache[toponym] = []
        return []

    results = [
        {
            "geonames_id": entry.get("geonameId"),
            "name": entry.get("name"),
            "lat": float(entry["lat"]) if entry.get("lat") else None,
            "lon": float(entry["lng"]) if entry.get("lng") else None,
            "feature_class": entry.get("fcl"),
            "feature_code": entry.get("fcode"),
            "country_code": entry.get("countryCode"),
            "admin1_name": entry.get("adminName1"),
        }
        for entry in payload.get("geonames", [])
    ]
    _geonames_cache[toponym] = results
    return results


## Load inputs

In [3]:
with open(INDEX_FILE, "rb") as f:
    by_uri, by_name = pickle.load(f)
by_name_keys = list(by_name.keys())

with jsonlines.open(NER_FILE) as reader:
    ner_results = list(reader)

with jsonlines.open(ARTICLES_FILE) as reader:
    articles_by_id = {a["id"]: a for a in reader}

logger.info("Loaded Pleiades index (%d places, %d name keys), %d NER results, %d articles",
            len(by_uri), len(by_name), len(ner_results), len(articles_by_id))


2026-06-08 17:32:05,602 [INFO] Loaded Pleiades index (42134 places, 59671 name keys), 2792 NER results, 1358 articles


## LLM arbitration prompts

Used in Pass 3 (choosing among multiple Pleiades candidates) and Pass 4 (validating a GeoNames candidate against the archaeological context).

In [4]:
PLEIADES_ARBITRATION_TEMPLATE = """\
Given this archaeological context, select the most appropriate
Pleiades entry for the toponym "{toponym}".

Context: {context}
Article abstract (excerpt): {abstract}

Candidates:
{candidates}

Return JSON: {{"selected_uri": "<uri or null>", "reason": "<brief reason>"}}
Select null if none of the candidates is a plausible match.
"""

GEONAMES_ARBITRATION_TEMPLATE = """\
Does any of these geographic entries correspond to the ancient
archaeological site mentioned in the context?

Context: {context}
Article abstract (excerpt): {abstract}

GeoNames candidates:
{candidates}

Return JSON:
{{"selected_geonames_id": <int or null>, "reason": "<brief reason>"}}
Return null if none are appropriate for an archaeological context.
"""

PLEIADES_ARBITRATION_SCHEMA = {
    "type": "object",
    "properties": {
        "selected_uri": {"type": ["string", "null"]},
        "reason": {"type": "string"},
    },
    "required": ["selected_uri", "reason"],
}

GEONAMES_ARBITRATION_SCHEMA = {
    "type": "object",
    "properties": {
        "selected_geonames_id": {"type": ["integer", "null"]},
        "reason": {"type": "string"},
    },
    "required": ["selected_geonames_id", "reason"],
}


def arbitrate(template: str, schema: dict, **kwargs) -> dict:
    """Send an arbitration prompt to the LLM and parse the structured JSON response."""
    response = client.chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": template.format(**kwargs)}],
        format=schema,
        options={"temperature": 0},
    )
    return json.loads(response["message"]["content"])


## Result record builder

Builds a fully-populated output record for a single toponym instance, given the resolution outcome of the four-pass pipeline.

In [5]:
def make_record(article_id, toponym, *, place=None, geonames=None,
                source="unresolved", method="unresolved", confidence="none"):
    """Assemble the output record for one toponym instance given its resolution outcome."""
    record = {
        "article_id": article_id,
        "toponym_raw": toponym["toponym_raw"],
        "toponym_normalized": toponym["toponym_normalized"],
        "entity_type": toponym["entity_type"],
        "context_fragment": toponym["context_fragment"],
        "ner_confidence": toponym["confidence"],
        "pleiades_uri": None,
        "pleiades_name": None,
        "pleiades_place_types": None,
        "geonames_id": None,
        "geonames_name": None,
        "gazetteer_source": source,
        "lat": None,
        "lon": None,
        "disambiguation_method": method,
        "disambiguation_confidence": confidence,
    }
    if place is not None:
        record.update({
            "pleiades_uri": place["uri"],
            "pleiades_name": place["title"],
            "pleiades_place_types": place["place_types"],
            "lat": place["lat"],
            "lon": place["lon"],
        })
    if geonames is not None:
        record.update({
            "geonames_id": geonames["geonames_id"],
            "geonames_name": geonames["name"],
            "lat": geonames["lat"],
            "lon": geonames["lon"],
        })
    return record


## Four-pass disambiguation

`disambiguate_toponym` runs the full pipeline for a single toponym instance and returns the finished output record.

In [6]:
def format_pleiades_candidates(places: list[dict]) -> str:
    """Render Pleiades candidates as a numbered list for the LLM prompt."""
    lines = []
    for place in places:
        lines.append(f"- name: {place['title']!r}, uri: {place['uri']}, "
                     f"place_types: {place['place_types']}, time_periods: {place['time_periods']}")
    return "\n".join(lines)


def format_geonames_candidates(results: list[dict]) -> str:
    """Render GeoNames candidates as a numbered list for the LLM prompt."""
    lines = []
    for r in results:
        lines.append(f"- name: {r['name']!r}, feature: {r['feature_class']}/{r['feature_code']}, "
                     f"country: {r['country_code']}, lat: {r['lat']}, lon: {r['lon']}, id: {r['geonames_id']}")
    return "\n".join(lines)


def disambiguate_toponym(toponym: dict, article: dict) -> dict:
    """Run the four-pass pipeline and return the disambiguation record for one toponym."""
    article_id = article["id"]
    abstract_excerpt = (article.get("abstract") or "")[:300]
    context = toponym["context_fragment"]
    key = normalize(toponym["toponym_normalized"])

    # --- Pass 1: exact match on Pleiades ---
    exact_matches = by_name.get(key, [])

    if len(exact_matches) == 1:
        return make_record(article_id, toponym, place=exact_matches[0],
                           source="pleiades", method="exact", confidence="high")

    candidates = None
    if len(exact_matches) == 0:
        # --- Pass 2: fuzzy match on Pleiades ---
        fuzzy_hits = process.extract(key, by_name_keys, score_cutoff=FUZZY_THRESHOLD, limit=FUZZY_CANDIDATES)
        fuzzy_places = [by_name[match_key][0] for match_key, _score, _idx in fuzzy_hits]

        if len(fuzzy_places) == 0:
            candidates = None  # skip Pass 3, go to Pass 4
        elif len(fuzzy_places) == 1:
            return make_record(article_id, toponym, place=fuzzy_places[0],
                               source="pleiades", method="fuzzy", confidence="medium")
        else:
            candidates = fuzzy_places
    else:
        # 2+ exact matches: skip Pass 2, go directly to Pass 3
        candidates = exact_matches[:FUZZY_CANDIDATES]

    # --- Pass 3: LLM arbitration on Pleiades candidates ---
    if candidates:
        try:
            arbitration = arbitrate(
                PLEIADES_ARBITRATION_TEMPLATE, PLEIADES_ARBITRATION_SCHEMA,
                toponym=toponym["toponym_normalized"], context=context, abstract=abstract_excerpt,
                candidates=format_pleiades_candidates(candidates),
            )
            selected_uri = arbitration.get("selected_uri")
        except Exception as exc:
            logger.warning("Pleiades arbitration failed for %r (article %s): %s",
                           toponym["toponym_normalized"], article_id, exc)
            selected_uri = None

        if selected_uri and selected_uri in by_uri:
            return make_record(article_id, toponym, place=by_uri[selected_uri],
                               source="pleiades", method="llm", confidence="low")

    # --- Pass 4: GeoNames fallback ---
    geonames_results = search_geonames(toponym["toponym_normalized"])
    if not geonames_results:
        return make_record(article_id, toponym)

    try:
        arbitration = arbitrate(
            GEONAMES_ARBITRATION_TEMPLATE, GEONAMES_ARBITRATION_SCHEMA,
            context=context, abstract=abstract_excerpt,
            candidates=format_geonames_candidates(geonames_results),
        )
        selected_id = arbitration.get("selected_geonames_id")
    except Exception as exc:
        logger.warning("GeoNames arbitration failed for %r (article %s): %s",
                       toponym["toponym_normalized"], article_id, exc)
        selected_id = None

    selected = next((r for r in geonames_results if r["geonames_id"] == selected_id), None)
    if selected is not None:
        return make_record(article_id, toponym, geonames=selected,
                           source="geonames", method="geonames", confidence="low")

    return make_record(article_id, toponym)


## Processing

Resume from `OUTPUT_FILE`: collect `article_id`s already written and skip them. Each toponym produces its own output record, written immediately.

In [7]:
def load_processed_article_ids(path: Path) -> set[int]:
    """Collect article_id values already present in the output file."""
    if not path.exists():
        return set()
    with jsonlines.open(path) as reader:
        return {row["article_id"] for row in reader}


OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
processed_article_ids = load_processed_article_ids(OUTPUT_FILE)
remaining = [r for r in ner_results if r["article_id"] not in processed_article_ids]

logger.info("%d articles in NER results, %d already processed, %d remaining",
            len(ner_results), len(processed_article_ids), len(remaining))

with jsonlines.open(OUTPUT_FILE, mode="a") as writer:
    for ner_result in tqdm(remaining, desc="Disambiguation"):
        article_id = ner_result["article_id"]
        article = articles_by_id.get(article_id)
        if article is None:
            logger.warning("Article %s not found in articles cache, skipping", article_id)
            continue

        for toponym in ner_result.get("toponyms", []):
            record = disambiguate_toponym(toponym, article)
            writer.write(record)


2026-06-08 17:32:05,620 [INFO] 2792 articles in NER results, 0 already processed, 2792 remaining
Disambiguation:   0%|          | 0/2792 [00:00<?, ?it/s]2026-06-08 17:32:13,469 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-08 17:32:23,570 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Disambiguation:   0%|          | 8/2792 [00:24<2:24:58,  3.12s/it]2026-06-08 17:32:36,466 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-08 17:32:45,477 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-08 17:32:49,803 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-08 17:32:59,095 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Disambiguation:   1%|          | 18/2792 [00:58<2:35:09,  3.36s/it]2026-06-08 17:33:09,850 [INFO] HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-06-08 17:

## Final summary

In [8]:
with jsonlines.open(OUTPUT_FILE) as reader:
    all_records = list(reader)

method_counts = Counter(r["disambiguation_method"] for r in all_records)
resolved = sum(1 for r in all_records if r["gazetteer_source"] in ("pleiades", "geonames"))
unresolved = sum(1 for r in all_records if r["gazetteer_source"] == "unresolved")
total = len(all_records)

print("Counts per disambiguation_method:")
for method, count in method_counts.most_common():
    print(f"  {method:<12} {count}")
print()
print(f"Resolved (pleiades + geonames): {resolved}")
print(f"Unresolved:                     {unresolved}")
print(f"Overall resolution rate:        {resolved / total:.1%}" if total else "No records.")


Counts per disambiguation_method:
  exact        605
  unresolved   288
  llm          276
  geonames     138
  fuzzy        1

Resolved (pleiades + geonames): 1020
Unresolved:                     288
Overall resolution rate:        78.0%
